# 소성가공 예지보전 LSTM-Autoencoder (Google Colab)

가이드북 2~6단계 노트북입니다. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU**를 고르고 셀을 위에서부터 실행하세요.

CSV 원본은 GitHub에 올리지 마세요. Google Drive의 `내 드라이브/kamp/fine-blanking-press/raw/`에 `press_data_normal.csv`, `outlier_data.csv` 두 파일을 두고, 결과는 Drive의 `artifacts/`에 저장합니다. TensorFlow 버전을 강제로 교체하지 않고 Colab 기본 버전을 사용해 실제 실행 버전을 기록합니다.


In [ ]:
# 1. 런타임 / TensorFlow 확인. 기본 이미지에 없을 때만 TF 2.20 CPU/GPU wheel을 설치한다.
import os, sys, subprocess
print("Python:", sys.version)
try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow==2.20.0"])
    import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "matplotlib", "joblib"])


In [ ]:
# 2. Google Drive 연결
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/kamp/fine-blanking-press")
DATA_DIR = DRIVE_PROJECT / "raw"
ARTIFACT_DIR = DRIVE_PROJECT / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("CSV 존재 여부:", {p.name: p.exists() for p in [DATA_DIR / "press_data_normal.csv", DATA_DIR / "outlier_data.csv"]})


위 CSV 존재 여부가 모두 `True`인지 확인하세요. `False`면 로컬의 두 CSV를 Google Drive의 `kamp/fine-blanking-press/raw` 폴더에 업로드한 다음 이 셀을 다시 실행하세요.


In [ ]:
# 3. 실행 파이프라인 준비: 코드와 결과는 Drive에, 실행 중인 복사본은 Colab VM에 둔다.
os.environ["PROJECT_ROOT"] = str(DRIVE_PROJECT)
pipeline_source = '"""KAMP 실습 2~6단계: inspect → prepare → train → verify.\n\n원본 CSV는 읽기만 하고 모든 산출물은 Git에서 제외된 artifacts 아래에 쓴다.\nTensorFlow는 학습/재로딩 시에만 불러오므로 설치 전에도 데이터 검사가 가능하다.\n"""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport time\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")  # GUI 없이 실행하고 그림은 PNG로 저장한다.\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn import metrics\nfrom sklearn.preprocessing import MinMaxScaler\n\n# Colab에서는 PROJECT_ROOT를 Google Drive 경로로 지정해 산출물을 런타임 종료 후에도 보존한다.\nROOT = Path(os.environ.get("PROJECT_ROOT", str(Path(__file__).resolve().parent)))\nFEATURES = ["AI0_Vibration", "AI1_Vibration", "AI2_Current"]\nSEQUENCE, HORIZON = 20, 100\n\n\ndef write_json(path, value):\n    """numpy 객체는 호출부에서 기본 Python 자료형으로 변환해 전달한다."""\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef read_data(data_dir):\n    """2단계: 스키마/라벨/시간/센서값 검사. 중복은 보고만 하고 원본을 보존한다."""\n    frames, report = {}, {}\n    for name, filename, count, label, date in [\n        ("normal", "press_data_normal.csv", 20000, 0, "2022-07-12"),\n        ("anomaly", "outlier_data.csv", 600, 1, "2022-07-17"),\n    ]:\n        path = data_dir / filename\n        df = pd.read_csv(path, index_col=0)\n        expected = ["TimeStamp"] + FEATURES + ["Equipment_state"]\n        if list(df.columns) != expected or len(df) != count:\n            raise ValueError(f"{filename}: 가이드의 열 구성 또는 행 수와 다릅니다.")\n        if df.isna().any().any():\n            raise ValueError(f"{filename}: 결측치를 확인하세요.")\n        duplicates = int(df.duplicated().sum())\n        df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], errors="raise")\n        for col in FEATURES + ["Equipment_state"]:\n            df[col] = pd.to_numeric(df[col], errors="raise")\n        if not np.isfinite(df[FEATURES].to_numpy()).all():\n            raise ValueError(f"{filename}: 센서에 NaN 또는 무한대가 있습니다.")\n        if not df["Equipment_state"].eq(label).all():\n            raise ValueError(f"{filename}: 정상/이상 라벨이 예상과 다릅니다.")\n        if not df.TimeStamp.is_monotonic_increasing:\n            raise ValueError(f"{filename}: 시간 역전이 있습니다. 원본 순서를 확인하세요.")\n        if not df.TimeStamp.dt.strftime("%Y-%m-%d").eq(date).all():\n            raise ValueError(f"{filename}: 수집 날짜가 가이드와 다릅니다.")\n        delta = df.TimeStamp.diff().dt.total_seconds().dropna()\n        # 시간 간격의 이상은 자동 보간하지 않는다. 가이드 재현에 미치는 영향을 보고한다.\n        report[name] = {\n            "rows": len(df), "duplicate_rows": duplicates,\n            "missing_cells": int(df.isna().sum().sum()),\n            "non_0_1_second_intervals": int((~np.isclose(delta, 0.1)).sum()),\n            "interval_counts": {str(k): int(v) for k, v in delta.value_counts().items()},\n            "sensor_statistics": df[FEATURES].describe().to_dict(),\n            "absolute_correlation": df[FEATURES].abs().corr().to_dict(),\n            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n        }\n        frames[name] = df\n    return frames, report\n\n\ndef plot_data(frames, out):\n    """원본과 절댓값 변환 후의 세 센서를 나란히 비교한다."""\n    for name, df in frames.items():\n        fig, axes = plt.subplots(3, 2, figsize=(13, 8))\n        for i, col in enumerate(FEATURES):\n            axes[i, 0].plot(df.TimeStamp, df[col], linewidth=0.5)\n            axes[i, 1].plot(df.TimeStamp, df[col].abs(), linewidth=0.5)\n            axes[i, 0].set_title(col + " / raw")\n            axes[i, 1].set_title(col + " / absolute")\n        fig.autofmt_xdate()\n        fig.tight_layout()\n        fig.savefig(out / f"{name}_signals.png")\n        plt.close(fig)\n\n\ndef windows(values, labels):\n    """3단계: 가이드의 N-20-100개 윈도와 미래 라벨 인덱스를 정확히 따른다.\n\n    입력은 [i:i+20], 라벨은 i+120이다. 라벨은 평가에만 쓰며 학습 목표는 X다.\n    일정 라벨 파일이므로 이 연결 자체가 10초 사전 예측을 입증하지는 않는다.\n    """\n    count = len(values) - SEQUENCE - HORIZON\n    if count <= 0:\n        raise ValueError("윈도를 생성하려면 구간당 120행보다 많은 데이터가 필요합니다.")\n    x = np.stack([values[i:i + SEQUENCE] for i in range(count)]).astype("float32")\n    y = np.asarray(labels[SEQUENCE + HORIZON:], dtype="int64")\n    return x, y\n\n\ndef prepare(frames, mode):\n    """guide는 책의 분할, strict는 원본 구간을 먼저 분할해 경계 중첩을 방지한다."""\n    normal, anomaly = frames["normal"], frames["anomaly"]\n    scaler = MinMaxScaler()\n    scaler.fit(normal[FEATURES].iloc[:15000].abs())\n\n    def make(df):\n        # 평가 센서값이 학습 범위를 넘으면 1보다 커질 수 있다. clip하지 않는다.\n        return windows(scaler.transform(df[FEATURES].abs()), df.Equipment_state.to_numpy())\n\n    train, _ = make(normal.iloc[:15000])\n    if mode == "guide":\n        nx, ny = make(normal.iloc[15000:])\n        ax, ay = make(anomaly)\n        vn, va = (nx[:880], ny[:880]), (ax[:300], ay[:300])\n        tn, ta = (nx[880:], ny[880:]), (ax[300:], ay[300:])\n    else:\n        # 각 구간 내에서만 입력과 미래 라벨을 구성하므로 서로 원본 행을 공유하지 않는다.\n        vn, tn = make(normal.iloc[15000:16000]), make(normal.iloc[16000:])\n        va, ta = make(anomaly.iloc[:300]), make(anomaly.iloc[300:])\n    data = {\n        "train": train, "valid_normal": vn[0],\n        "valid": np.concatenate([vn[0], va[0]]), "valid_y": np.concatenate([vn[1], va[1]]),\n        "test": np.concatenate([tn[0], ta[0]]), "test_y": np.concatenate([tn[1], ta[1]]),\n    }\n    return data, scaler\n\n\ndef tensorflow():\n    try:\n        import tensorflow as tf\n    except ImportError as exc:\n        raise RuntimeError("TensorFlow를 현재 인터프리터에 설치하세요. README의 환경 설정을 참고하세요.") from exc\n    return tf\n\n\ndef build_model(tf):\n    """4단계: 20×3 입력을 압축한 뒤 같은 크기로 복원하는 LSTM-Autoencoder."""\n    layers = tf.keras.layers\n    model = tf.keras.Sequential([\n        layers.Input(shape=(SEQUENCE, len(FEATURES))),\n        layers.LSTM(64, return_sequences=True),\n        layers.LSTM(32),\n        layers.RepeatVector(SEQUENCE),\n        layers.LSTM(32, return_sequences=True),\n        layers.LSTM(64, return_sequences=True),\n        layers.TimeDistributed(layers.Dense(len(FEATURES))),\n    ])\n    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse")\n    return model\n\n\ndef score(model, x):\n    """가이드처럼 마지막 시점의 세 센서만 평균해 윈도별 재구성 오차를 구한다."""\n    predicted = model.predict(x, batch_size=128, verbose=0)\n    result = np.mean(np.square(x[:, -1, :] - predicted[:, -1, :]), axis=1)\n    if not np.isfinite(result).all():\n        raise ValueError("재구성 오차가 유한하지 않습니다. 학습 상태를 확인하세요.")\n    return result\n\n\ndef select_threshold(y, scores, method):\n    """5단계: 검증셋만 사용한다. PR 곡선의 마지막 원소에는 대응 임곗값이 없다."""\n    precision, recall, thresholds = metrics.precision_recall_curve(y, scores)\n    p, r = precision[:-1], recall[:-1]\n    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) != 0)\n    equal = np.flatnonzero(p == r)\n    index = int(equal[0]) if method == "guide" and len(equal) else int(np.argmax(f1))\n    # sklearn PR의 >= 판정과 평가 판정을 통일한다(책의 >와 경계 처리 차이).\n    return float(thresholds[index]), {\n        "requested_method": method,\n        "used_method": "precision_equals_recall" if method == "guide" and len(equal) else "max_f1",\n        "comparison": ">=", "precision": float(p[index]), "recall": float(r[index]),\n    }\n\n\ndef evaluate(y, scores, threshold):\n    pred = (scores >= threshold).astype(int)\n    tn, fp, fn, tp = metrics.confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return {\n        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],\n        "accuracy": float(metrics.accuracy_score(y, pred)),\n        "precision": float(metrics.precision_score(y, pred, zero_division=0)),\n        "recall": float(metrics.recall_score(y, pred, zero_division=0)),\n        "f1": float(metrics.f1_score(y, pred, zero_division=0)),\n        "false_positive_rate": float(fp / (tn + fp)),\n    }\n\n\ndef verify(data_dir, out):\n    """6단계: 저장된 모델·scaler·설정으로 원본부터 재처리하여 이전 점수와 비교한다.\n\n    joblib 파일은 자신이 생성한 로컬 산출물만 읽는다.\n    테스트 정답으로 임곗값을 재학습하지 않는다.\n    """\n    tf = tensorflow()\n    config = json.loads((out / "config.json").read_text(encoding="utf-8"))\n    if config["features"] != FEATURES or config["sequence"] != SEQUENCE or config["horizon"] != HORIZON:\n        raise ValueError("저장 설정과 현재 코드의 입력 규격이 다릅니다.")\n    frames, report = read_data(data_dir)\n    if {k: v["sha256"] for k, v in report.items()} != config["data_sha256"]:\n        raise ValueError("학습 시점과 원본 데이터가 다릅니다.")\n    scaler = joblib.load(out / "scaler.joblib")\n    # prepare를 다시 호출해 scaler를 재학습하지 않고 저장된 scaler만 사용한다.\n    if config["split"] == "guide":\n        segments = [(frames["normal"].iloc[15000:], 880), (frames["anomaly"], 300)]\n    else:\n        segments = [(frames["normal"].iloc[16000:], 0), (frames["anomaly"].iloc[300:], 0)]\n    xs = []\n    for df, skip in segments:\n        x, _ = windows(scaler.transform(df[FEATURES].abs()), df.Equipment_state.to_numpy())\n        xs.append(x[skip:])\n    model = tf.keras.models.load_model(out / "model.h5", compile=False)\n    restored = score(model, np.concatenate(xs))\n    original = np.load(out / "test_scores.npy", allow_pickle=False)\n    np.testing.assert_allclose(restored, original, rtol=1e-5, atol=1e-7)\n    np.testing.assert_array_equal(restored >= config["threshold"], original >= config["threshold"])\n    result = {"passed": True, "samples": len(restored), "max_absolute_error": float(np.max(np.abs(restored - original)))}\n    write_json(out / "reload_check.json", result)\n    print("재로딩 검증:", result)\n\n\ndef train(data, scaler, report, args, out):\n    tf = tensorflow()\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    tf.random.set_seed(args.seed)\n    model = build_model(tf)\n    model.summary()\n    callbacks = [\n        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.7, patience=50),\n        tf.keras.callbacks.EarlyStopping(monitor="val_loss", min_delta=0.00001, patience=120, restore_best_weights=True),\n        tf.keras.callbacks.TerminateOnNaN(),\n    ]\n    started = time.perf_counter()\n    history = model.fit(\n        data["train"], data["train"], epochs=args.epochs, batch_size=128,\n        validation_data=(data["valid_normal"], data["valid_normal"]), callbacks=callbacks,\n        shuffle=True, verbose=2,\n    )\n    if not all(np.isfinite(values).all() for values in history.history.values()):\n        raise ValueError("학습 loss 또는 기록에 NaN/무한대가 있습니다.")\n    threshold, selection = select_threshold(data["valid_y"], score(model, data["valid"]), args.threshold_method)\n    scores = score(model, data["test"])\n    result = evaluate(data["test_y"], scores, threshold)\n    config = {\n        "features": FEATURES, "absolute_value": True, "sequence": SEQUENCE, "horizon": HORIZON,\n        "score": "last_timestep_mse", "split": args.split, "seed": args.seed,\n        "threshold": threshold, "threshold_selection": selection,\n        "epochs_requested": args.epochs, "epochs_completed": len(history.history["loss"]),\n        "seconds": time.perf_counter() - started,\n        "data_sha256": {k: v["sha256"] for k, v in report.items()},\n        "python": platform.python_version(),\n        "versions": {p: importlib.metadata.version(p) for p in ["tensorflow", "numpy", "pandas", "scikit-learn", "matplotlib", "joblib"]},\n    }\n    # HDF5는 TF 2.7과 최신 Keras에서 모두 읽을 수 있도록 선택했다.\n    model.save(out / "model.h5", include_optimizer=False)\n    joblib.dump(scaler, out / "scaler.joblib")\n    np.save(out / "test_scores.npy", scores)\n    write_json(out / "config.json", config)\n    write_json(out / "metrics.json", result)\n    write_json(out / "history.json", {k: [float(v) for v in values] for k, values in history.history.items()})\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n    axes[0].plot(history.history["loss"], label="train")\n    axes[0].plot(history.history["val_loss"], label="validation")\n    axes[0].legend()\n    axes[0].set_title("Reconstruction loss")\n    axes[1].scatter(np.arange(len(scores)), scores, c=data["test_y"], s=3)\n    axes[1].axhline(threshold, color="red")\n    axes[1].set_title("Test score / threshold")\n    metrics.ConfusionMatrixDisplay(np.array(result["confusion_matrix"]), display_labels=["Normal", "Anomaly"]).plot(ax=axes[2], colorbar=False)\n    fig.tight_layout()\n    fig.savefig(out / "evaluation.png")\n    plt.close(fig)\n    print("평가 결과:", result)\n    verify(args.data_dir, out)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("command", choices=["inspect", "prepare", "train", "verify"])\n    parser.add_argument("--data-dir", type=Path, default=Path(os.environ.get("DATA_DIR", str(Path.home() / "datasets/kamp/fine-blanking-press/raw"))))\n    parser.add_argument("--run", default="smoke", help="artifacts 아래 실행 폴더 이름")\n    parser.add_argument("--split", choices=["guide", "strict"], default="guide")\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--threshold-method", choices=["guide", "f1"], default="guide")\n    args = parser.parse_args()\n    if args.epochs < 1 or not args.run or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in args.run):\n        parser.error("epochs는 양수, run은 영문/숫자/밑줄/하이픈 이름이어야 합니다.")\n    out = ROOT / "artifacts" / args.run\n    if args.command == "verify":\n        verify(args.data_dir, out)\n        return\n    # 이전 학습 결과를 덮어쓰지 않도록 실행마다 새 이름을 사용한다.\n    if (out / "config.json").exists():\n        parser.error("이미 학습된 실행 폴더입니다. 다른 --run 이름을 사용하세요.")\n    out.mkdir(parents=True, exist_ok=True)\n    frames, report = read_data(args.data_dir)\n    write_json(out / "quality.json", report)\n    plot_data(frames, out)\n    print("데이터 검사:", {k: {f: v[f] for f in ["rows", "duplicate_rows", "non_0_1_second_intervals"]} for k, v in report.items()})\n    if args.command == "inspect":\n        return\n    data, scaler = prepare(frames, args.split)\n    shapes = {k: list(v.shape) for k, v in data.items()}\n    write_json(out / "shapes.json", {"split": args.split, "shapes": shapes})\n    print("데이터 구성:", shapes)\n    if args.command == "train":\n        train(data, scaler, report, args, out)\n\n\nif __name__ == "__main__":\n    main()\n'
Path("/content/press_pipeline.py").write_text(pipeline_source, encoding="utf-8")
sys.path.insert(0, "/content")
import press_pipeline as pp
print("준비 완료:", pp.__file__)


In [ ]:
# 4. 데이터 품질 검사 및 그래프 생성
RUN_NAME = "colab-inspect"
out = ARTIFACT_DIR / RUN_NAME
out.mkdir(parents=True, exist_ok=True)
frames, quality = pp.read_data(DATA_DIR)
pp.write_json(out / "quality.json", quality)
pp.plot_data(frames, out)
print({name: {k: report[k] for k in ["rows", "duplicate_rows", "non_0_1_second_intervals"]} for name, report in quality.items()})


In [ ]:
# 5. 전처리와 윈도 구성 확인
SPLIT = "guide"  # 추가 실험은 "strict"로 변경하고 별도 실행 이름을 사용할 것
data, scaler = pp.prepare(frames, SPLIT)
print({name: values.shape for name, values in data.items()})


In [ ]:
# 6. 짧은 시험 학습: 3 epoch로 학습-평가-저장-재로딩까지 연결 확인
smoke_args = type("Args", (), {"seed": 42, "epochs": 3, "split": SPLIT, "threshold_method": "guide", "data_dir": DATA_DIR})()
smoke_out = ARTIFACT_DIR / "colab-smoke"
smoke_out.mkdir(parents=True, exist_ok=True)
pp.train(data, scaler, quality, smoke_args, smoke_out)


## 본 학습

시험 학습이 성공한 뒤 실행하세요. 최대 800 epoch이며 조기 종료할 수 있습니다. Colab 세션이 끊길 수 있으므로 학습 결과는 Google Drive에 저장합니다.


In [ ]:
# 7. 가이드 분할 본 학습
full_args = type("Args", (), {"seed": 42, "epochs": 800, "split": SPLIT, "threshold_method": "guide", "data_dir": DATA_DIR})()
full_out = ARTIFACT_DIR / "colab-guide-full"
full_out.mkdir(parents=True, exist_ok=True)
pp.train(data, scaler, quality, full_args, full_out)


In [ ]:
# 8. 저장된 모델과 scaler 재로딩 검증
pp.verify(DATA_DIR, ARTIFACT_DIR / "colab-guide-full")


## 선택 실험: strict 분할

가이드 재현과 구분되는 실험입니다. 원본 구간을 먼저 나눠 경계 윈도 중첩을 줄이고 검증 F1 최대 지점으로 임곗값을 정합니다. 결과는 별도 폴더에 저장됩니다.


In [ ]:
strict_data, strict_scaler = pp.prepare(frames, "strict")
print({name: values.shape for name, values in strict_data.items()})
strict_args = type("Args", (), {"seed": 42, "epochs": 800, "split": "strict", "threshold_method": "f1", "data_dir": DATA_DIR})()
strict_out = ARTIFACT_DIR / "colab-strict-full"
strict_out.mkdir(parents=True, exist_ok=True)
pp.train(strict_data, strict_scaler, quality, strict_args, strict_out)


평가 결과, 설정과 학습 이력은 Drive의 `artifacts/<실행이름>/`에서 확인하세요. 기준 guide shape는 train `(14880,20,3)`, valid `(1180,20,3)`, test `(4180,20,3)`입니다. 데이터 시간 간격은 보간하지 않습니다. 현재 파일처럼 정상/이상 라벨이 각 파일에서 일정하면 이 실험만으로 고장 10초 전 탐지 성능을 입증할 수 없습니다. Colab의 Python/TensorFlow/GPU 가용성은 런타임마다 달라지며 실제 버전은 결과 설정에 기록됩니다.